# RADAR vs LiDAR: Recommendations and Conclusion

## Recommendations

The following recommendations are based on two complementary analyses conducted using the selected MAN TruckScenes trainval subset (sensor data packages 01, 03 and 05):

- **Object Detection Analysis:** Evaluation of OmniHD-pretrained LiDAR and 4D RADAR PointPillars models on 10,063 TruckScenes samples containing 87,781 ground-truth objects, without retraining or fine-tuning.
- **Condition-Based Velocity Analysis:** Evaluation of LiDAR-derived and RADAR-reconstructed 2D ground-plane velocity, including estimation accuracy, availability, operating conditions and simple sensor fusion.

These analyses evaluate different tasks and experimental units. Object-detection performance should therefore be distinguished from velocity-estimation performance. The findings describe the performance of the current pipelines rather than the intrinsic capabilities of either sensor.

### Q1. Which sensor should be used as the primary source for object detection and 2D velocity estimation?

**Recommendation**

LiDAR should be used as the primary source for object detection, spatial localisation and reconstructed 2D vehicle velocity under the current pipeline. RADAR should be retained as a complementary sensor rather than used as a direct replacement for LiDAR.

**Evidence**

Object-detection performance:

| Metric | LiDAR | 4D RADAR |
|---|---:|---:|
| mAP | 17.51% | 0.63% |
| Precision | 80.16% | 35.99% |
| Recall | 39.09% | 3.89% |
| F1-score | 52.55% | 7.02% |
| Average Translation Error (ATE) | 0.442 m | 1.024 m |

Velocity-estimation performance:

| Metric | LiDAR | RADAR |
|---|---:|---:|
| Median absolute speed error | 0.443 m/s | 0.572 m/s |
| P90 absolute speed error | 2.071 m/s | 30.469 m/s |
| Median direction error | 0.433° | 6.253° |
| Stable 2D velocity availability | 46.9% | 18.5% |

Velocity accuracy was evaluated on intervals where both sensors produced valid estimates. Availability was measured over the reference intervals, with RADAR estimates required to satisfy the current numerical stability criterion.

**Interpretation**

LiDAR substantially outperforms RADAR in standalone object detection, achieving approximately ten times higher recall and considerably more accurate object localisation.

For reconstructed 2D velocity, the difference in median speed error is relatively small. However, RADAR's P90 error is approximately 14.7 times higher than LiDAR's, indicating a much greater risk of large estimation errors.

LiDAR also provides usable reconstructed 2D velocity more frequently and has substantially lower motion-direction error.

These findings consistently support LiDAR as the more reliable primary source for the two tasks evaluated.

**Limitations**

The object-detection models were pretrained on OmniHD and transferred directly to TruckScenes without retraining. The measured performance gap may therefore reflect model adaptation and sensor-configuration differences in addition to sensor characteristics.

The velocity findings evaluate the current reconstruction pipelines, not the intrinsic accuracy of the sensors' raw measurements.

### Q2. In which situations should RADAR be prioritised as a complementary object-detection source?

**Recommendation**

RADAR should be investigated as a supplementary detection source, particularly for moving objects, large vehicles and objects at 50–75 m, where its current model demonstrates the strongest complementary contribution to LiDAR.

**Evidence**

Across 87,781 ground-truth objects:

| Detection outcome | Number of objects |
|---|---:|
| Detected by both sensors | 2,418 |
| Detected by LiDAR only | 31,892 |
| Detected by RADAR only | 996 |
| Missed by both sensors | 52,475 |

RADAR recovered 996 objects that LiDAR missed.

This corresponds to:

- **1.86% of LiDAR's missed objects**.
- An increase in potential combined detection recall from **39.09% to 40.22%**, equivalent to **1.13 percentage points**.

Complementarity is concentrated in specific conditions:

| Condition | RADAR-only detections | LiDAR recall | Potential combined recall | LiDAR-miss recovery rate |
|---|---:|---:|---:|---:|
| Large vehicles | 719 | 10.7% | 14.1% | 3.8% |
| Moving objects | 994 | 45.0% | 46.5% | 2.7% |
| 50–75 m | 412 | 29.6% | 32.1% | 3.5% |

Among the 996 RADAR-only detections:

- **72.2% (719/996)** were large vehicles.
- **99.8% (994/996)** were moving objects.

**Interpretation**

Although RADAR has low standalone detection performance, its additional detections are not distributed uniformly across object types and conditions.

Large vehicles account for the majority of RADAR-only detections. Nearly all RADAR-only detections also involve moving objects, suggesting that its complementary detection contribution is concentrated on moving traffic participants.

The 50–75 m distance interval demonstrates a particularly useful gain, increasing potential recall by approximately 2.5 percentage points.

These results justify prioritising these conditions when investigating RADAR-assisted object detection, rather than assuming that RADAR contributes equally across all scenarios.

**Limitations**

The complementary recall is calculated from ground-truth-level detection outcomes. It represents the potential coverage of an ideal combination of the two detectors, not the performance of an implemented detection-fusion algorithm.

Simply adding RADAR predictions may introduce additional false positives. Therefore, improvements in overall precision, mAP or downstream driving performance have not been demonstrated.

Furthermore, the percentages 72.2% and 99.8% describe the composition of RADAR-only detections, not RADAR's detection success rate across all large or moving objects.

### Q3. How should RADAR's motion information be incorporated into future object-tracking systems?

**Recommendation**

Future work should investigate using RADAR's direct radial-velocity measurements to complement LiDAR-based object tracking, rather than relying on RADAR's independently reconstructed full-2D velocity as the primary motion estimate.

**Evidence**

The condition-based velocity analysis shows:

| Metric | LiDAR | RADAR |
|---|---:|---:|
| Stable 2D velocity availability | 46.9% | 18.5% |
| Median speed error | 0.443 m/s | 0.572 m/s |
| P90 speed error | 2.071 m/s | 30.469 m/s |

RADAR achieves approximately **47.3% full-rank 2D velocity reconstructability**, but only **18.5%** of reference intervals pass the numerical stability criterion.

The analysis also identifies selected conditions where RADAR produces competitive median speed errors:

| Condition | RADAR median error | LiDAR median error |
|---|---:|---:|
| 0–25 m | 0.301 m/s | 0.400 m/s |
| City | 0.234 m/s | 0.292 m/s |
| Parking | 0.092 m/s | 0.128 m/s |
| Train | 0.161 m/s | 0.517 m/s |

However, lower median error does not imply greater overall reliability. At 0–25 m, RADAR still has a P90 speed error of approximately **6.16 m/s**, compared with **2.07 m/s** for LiDAR.

**Interpretation**

The large difference between RADAR reconstructability and stable availability demonstrates a limitation of the current full-2D velocity reconstruction pipeline.

RADAR directly measures radial velocity, which describes motion along the sensor's line of sight. Reconstructing complete 2D motion requires additional assumptions and sufficient independent measurement constraints.

Instead of requiring RADAR to independently reconstruct both velocity components, a future tracking system could investigate using its radial-velocity observations alongside LiDAR-based object positions and temporal tracking information.

This approach may preserve useful motion measurements while avoiding some limitations associated with independent full-2D reconstruction.

**Limitations**

The current condition-based analysis evaluates reconstructed 2D velocity, not the standalone accuracy of direct RADAR radial-velocity measurements.

Therefore, the results do not establish that RADAR radial velocity is more accurate than LiDAR-derived velocity or that integrating radial velocity into an object tracker will improve performance.

The proposed tracking approach remains a future research direction and requires separate experimental validation.

### Q4. Should RADAR be prioritised in bad weather or when object visibility is reduced?

**Recommendation**

RADAR should not automatically be prioritised over LiDAR solely because of adverse weather or reduced object visibility. Sensor-use decisions should be based on demonstrated detection and velocity reliability rather than general assumptions about RADAR's physical resistance to weather.

**Evidence**

Object-detection recall by weather:

| Weather | LiDAR recall | RADAR recall |
|---|---:|---:|
| Clear | 41.2% | 6.7% |
| Fog | 44.0% | 1.0% |
| Rain | 45.5% | 0.9% |
| Snow | 13.5% | 2.2% |

In fog, LiDAR substantially outperforms RADAR for detection.

The velocity analysis also shows:

| Fog-related velocity metric | LiDAR | RADAR |
|---|---:|---:|
| Full-rank reconstructability | N/A | 51.1% |
| Stable velocity availability | 31.1% | 24.0% |
| Median speed error | 0.863 m/s | 5.401 m/s |
| P90 speed error | 2.874 m/s | 52.800 m/s |

Although RADAR can reconstruct a full-rank 2D velocity estimate for 51.1% of reference intervals in fog, only 24.0% satisfy the stability requirement. Its reconstructed velocity errors are also substantially larger than LiDAR's.

Reduced object visibility produces a similar pattern in the detection analysis:

- At Visibility Level 1 (0–40%), LiDAR recall is **14.9%**, compared with **1.0% for RADAR**.
- RADAR recovers only **0.8% of LiDAR misses** at Visibility Level 1.
- RADAR recovery reaches **2.7%** at Visibility Level 4 (81–100%), where objects are most visible.

**Interpretation**

RADAR's physical sensing characteristics may offer advantages in adverse weather, but these advantages are not demonstrated as superior object-detection accuracy or stable reconstructed 2D velocity under the current experimental setup.

The distinction between reconstructability, stable availability and estimation accuracy is particularly important. A sensor may produce more candidate measurements without producing more reliable final estimates.

Therefore, weather and visibility should be considered contextual factors rather than automatic rules for selecting RADAR over LiDAR.

**Limitations**

Weather comparisons are observational and may be affected by differences in scene composition, object classes, distances and motion characteristics.

The results also reflect the current pretrained detection models and velocity reconstruction methods. They should not be interpreted as evidence that RADAR has no intrinsic advantage in adverse weather.

### Q5. Should RADAR and LiDAR velocity estimates be combined through simple fusion?

**Recommendation**

Unconditional equal-weight fusion should not be used as the default method for combining RADAR and LiDAR reconstructed 2D velocities. Future work should investigate quality-aware fusion that accounts for numerical stability, cross-sensor agreement and measurement uncertainty.

**Evidence**

The condition-based velocity analysis evaluates simple equal-weight fusion on paired velocity estimates.

| Metric | LiDAR | RADAR | Fused |
|---|---:|---:|---:|
| Median speed error (m/s) | 0.443 | 0.572 | 0.556 |
| P90 speed error (m/s) | 2.071 | 30.469 | 21.697 |
| Median direction error (°) | 0.433 | 6.253 | 2.887 |

The fused estimate performs better than standalone RADAR but remains worse than LiDAR across all three reported metrics.

In particular, the fused P90 speed error of **21.697 m/s** is approximately 10.5 times higher than LiDAR's **2.071 m/s**.

However, RADAR provides additional availability:

- LiDAR-only velocity availability: **46.9%**.
- Combined LiDAR-or-RADAR availability: **48.3%**.
- Additional RADAR-only intervals: **2,955**.
- Absolute availability improvement: approximately **1.4 percentage points**.

**Interpretation**

The experiment demonstrates that simply averaging two sensor estimates does not necessarily improve accuracy.

When RADAR produces large velocity errors, combining its estimate equally with a more accurate LiDAR estimate can degrade the final result.

Nevertheless, RADAR provides a measurable coverage benefit by producing estimates for some intervals where LiDAR is unavailable.

These results suggest that a useful fusion strategy must distinguish between adding information and adding reliable information.

A potential future approach is to evaluate RADAR's numerical stability, agreement with LiDAR and estimated uncertainty before deciding whether to accept, reject or down-weight its contribution.

**Limitations**

Only simple equal-weight velocity fusion was evaluated. Quality-aware or adaptive fusion has not yet been implemented and should not be described as a demonstrated improvement.

The availability gain does not establish that every additional RADAR-only estimate is sufficiently accurate for autonomous-driving decisions.

Furthermore, these results concern reconstructed 2D velocity fusion, not object-detection fusion. No complete object-detection fusion pipeline was evaluated.

### Q6. Can RADAR provide more accurate object orientation information than LiDAR?

**Recommendation**

RADAR should be investigated as a complementary source of object orientation information, particularly for moving vehicles and objects at greater distances, where its successfully matched detections demonstrate lower Average Orientation Error (AOE) than LiDAR.

Rather than replacing LiDAR for object detection and localisation, future work should investigate whether RADAR's orientation estimates can improve the accuracy of LiDAR-supported object perception.

**Evidence**

The object-detection analysis shows that RADAR achieves lower AOE than LiDAR across several conditions.

| Condition | LiDAR AOE (rad) | RADAR AOE (rad) |
|---|---:|---:|
| Overall | 0.153 | **0.054** |
| Cars | 0.148 | **0.037** |
| Large vehicles | 0.226 | **0.091** |
| Moving objects | 0.127 | **0.053** |
| 0–25 m | 0.128 | **0.058** |
| 25–50 m | 0.151 | **0.053** |
| 50–75 m | 0.228 | **0.048** |

A particularly interesting finding is the difference in orientation error across distance.

For LiDAR, AOE increases from **0.128 rad at 0–25 m** to **0.228 rad at 50–75 m**.

For RADAR, AOE remains relatively stable, ranging from **0.048 to 0.058 rad** across the same distance intervals.

This suggests that RADAR's successfully matched detections maintain relatively accurate bounding-box orientation estimates even at greater distances.

**Interpretation**

Although LiDAR substantially outperforms RADAR in overall object detection, RADAR demonstrates lower orientation error among the objects it successfully detects.

This suggests a potential complementary role for RADAR beyond simply recovering objects missed by LiDAR.

For example, LiDAR could provide the primary object location and bounding box, while RADAR's orientation estimate could be considered as additional information when both sensors detect the same object.

This may be particularly relevant for moving vehicles and objects farther from the ego vehicle, where accurate orientation information could support downstream object tracking and scene understanding.

**Limitations**

The AOE results are calculated only for successfully matched detections. LiDAR detects **34,310 objects**, compared with only **3,414 for RADAR**, meaning the reported orientation errors are evaluated on different sets of objects.

Therefore, lower RADAR AOE does not conclusively establish that RADAR estimates orientation more accurately than LiDAR for identical targets.

To validate this finding, future work should compare orientation errors on the **2,418 ground-truth objects detected by both sensors**, using a paired evaluation.

Furthermore, AOE measures **bounding-box orientation**, not the direction of an object's actual movement. The lower RADAR AOE should therefore not be interpreted as evidence of superior velocity-direction estimation.

The current project has not tested an orientation-fusion system, so improvements to tracking or scene understanding remain potential applications rather than demonstrated outcomes.


## Conclusion

This project investigated the complementary roles of LiDAR and 4D RADAR in autonomous trucking through object-detection and reconstructed 2D velocity analyses using the selected MAN TruckScenes trainval subset.

The results demonstrate that **LiDAR is the more reliable primary source under the current experimental pipelines**, while RADAR provides specific complementary benefits that warrant further investigation.

**LiDAR performs substantially better in standalone object detection.** The LiDAR PointPillars model achieved **17.51% mAP, 80.16% precision and 39.09% recall**, compared with **0.63% mAP, 35.99% precision and 3.89% recall** for RADAR. LiDAR also produced lower object-centre localisation error.

**LiDAR provides more reliable reconstructed 2D velocity.** Although the paired median speed errors were relatively close (**0.443 m/s for LiDAR and 0.572 m/s for RADAR**), the P90 errors differed substantially (**2.071 m/s vs 30.469 m/s**). LiDAR also achieved higher velocity availability and much lower motion-direction error. These findings make LiDAR the stronger baseline for stable velocity estimation.

**RADAR contributes additional detections in specific conditions.** RADAR detected **996 ground-truth objects missed by LiDAR**, increasing potential combined recall from **39.09% to 40.22%**. Its complementary detections were concentrated on moving objects, large vehicles and the 50–75 m distance interval. In particular, RADAR recovered **719 large vehicles**, **994 moving objects** and **412 objects at 50–75 m** that LiDAR missed.

These results support investigating RADAR as a supplementary detection source, particularly where its contribution is most apparent. However, the reported recall improvement represents potential combined ground-truth coverage rather than the performance of an implemented detection-fusion system.

**RADAR's direct motion information remains a potential advantage, but its current full-2D reconstruction has significant limitations.** Although approximately **47.3%** of reference intervals allow full-rank RADAR velocity reconstruction, only **18.5%** produce estimates that satisfy the current stability criterion. Future work should investigate whether RADAR's direct radial-velocity measurements can contribute useful information within LiDAR-supported object tracking. This approach has not yet been experimentally validated.

**The current analyses do not demonstrate an overall RADAR advantage in adverse weather or reduced visibility.** LiDAR retains substantially higher object-detection recall in fog, rain and snow. In fog, RADAR also exhibits much larger reconstructed velocity errors. Consequently, weather resistance alone should not be used to justify automatic RADAR prioritisation.

**Simple sensor fusion does not necessarily improve performance.** The evaluated equal-weight velocity fusion achieved a median speed error of **0.556 m/s** and a P90 error of **21.697 m/s**, both worse than LiDAR alone. This supports investigating quality-aware fusion rather than unconditional averaging.

Finally, **RADAR's lower orientation error among matched detections is promising but requires further validation**. A paired evaluation on jointly detected objects, together with TruckScenes-specific RADAR fine-tuning and feature-adaptation experiments, would provide stronger evidence about its achievable performance.

### Final Recommendation

Based on the current experimental evidence, **LiDAR should remain the primary sensor for object detection, spatial localisation and reconstructed 2D velocity estimation**, while **4D RADAR should be retained as a complementary source of object detections and motion-related information**.

RADAR-assisted detection should be investigated first for moving objects, large vehicles and targets at 50–75 m, where the strongest additional detection coverage was observed. Future tracking research should evaluate whether direct RADAR radial-velocity measurements improve LiDAR-supported motion estimation, while fusion development should prioritise measurement-quality checks over unconditional averaging.

Before making stronger conclusions about RADAR's capabilities, the project should validate orientation accuracy on identical detected objects and investigate the effects of TruckScenes-specific model adaptation.

Overall, the findings support **a LiDAR-primary, RADAR-complementary approach**, with the role of each sensor determined by measured reliability and demonstrated contributions rather than general assumptions about sensor strengths.